# MSSQLDB Manager Usage Examples

This notebook demonstrates a practical `Microsoft SQL Server` workflow with `py-dockerdb`, focusing on fast local setup, predictable lifecycle control, and reproducible execution from a single Python interface. Instead of manually wiring container commands for each run, you configure once, start the service, validate connectivity, and then execute database operations in a consistent sequence.

If you are comparing engines, this pattern helps you isolate query behavior and schema differences without rewriting orchestration code. Official reference: https://learn.microsoft.com/sql/sql-server/.

## Table Of Contents

- [Table Of Contents](#table-of-contents)
- [Prerequisites](#prerequisites)
- [1. Setup](#1-setup)
  - [Import Dependencies](#import-dependencies)
- [2. Creating a SQL Server Database Instance](#2-creating-a-sql-server-database-instance)
- [3. Start the Database](#3-start-the-database)
- [4. Connect and Run SQL Queries](#4-connect-and-run-sql-queries)
  - [Creating a Table](#creating-a-table)
  - [Insert Data](#insert-data)
  - [Query Data](#query-data)
  - [Run More Complex Queries](#run-more-complex-queries)
- [5. Using Regular Python to Access the Database](#5-using-regular-python-to-access-the-database)
- [6. Using SQL Server-Specific Features](#6-using-sql-server-specific-features)
- [7. Clean Up](#7-clean-up)
- [8. Conclusion](#8-conclusion)
- [9. Conclusion](#9-conclusion)

## Prerequisites

No environment variables are required unless you intentionally override the defaults used in this notebook.

Additional prerequisites:
- Docker must be installed and running before you call `create_db()`.
- Install project dependencies (for example, `pip install -e ".[test]"` or `pip install py-dockerdb`).
- Run cells top-to-bottom so container lifecycle state remains consistent.

Provider documentation: https://learn.microsoft.com/sql/sql-server/
Typical setup guide: https://learn.microsoft.com/sql/linux/quickstart-install-connect-docker
Docker image setup reference: https://hub.docker.com/_/microsoft-mssql-server

Example datasets you can use to populate this database:
- AdventureWorks sample database: https://learn.microsoft.com/sql/samples/adventureworks-install-configure
- Wide World Importers sample database: https://learn.microsoft.com/sql/samples/wide-world-importers-what-is

## 1. Setup

### Install Required Packages

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
!pip install ipython-sqlcmd py-dockerdb pyodbc

### Import Dependencies

This subsection details `import dependencies` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
import uuid
from pathlib import Path
from docker_db.dbs.mssql_db import MSSQLConfig, MSSQLDB

# For SQL cell magic
%load_ext sqlcmd

## 2. Creating a SQL Server Database Instance

Let's create a temporary directory for our database files:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
import tempfile
temp_dir = Path(tempfile.mkdtemp())
container_name = f"demo-mssql-{uuid.uuid4().hex[:8]}"
init_script_path = Path("configs", "mssql", "initdb.sql")
init_script_path.exists()

This step executes code block `7` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
from utils import display_sql_script
display_sql_script(init_script_path)

Now, let's set up the MSSQLDB configuration:

This context is intentionally explicit so the next code cell is not a blind execution step and you can quickly diagnose issues if behavior differs from expectations.

In [ ]:
# Generate a unique container name
container_name = f"demo-mssql-{uuid.uuid4().hex[:8]}"

# Create a configuration for our database
config = MSSQLConfig(
    user="demouser",
    host="127.0.0.1",               # <- for some reason, this is not working with 'localhost'
    password="Demo_Pass123",
    database="demodb",
    sa_password="StrongPass123!",
    project_name="demo",
    workdir=temp_dir,
    container_name=container_name,
    retries=20,
    delay=3,
    init_script=init_script_path,
    env_vars={"YourEnvVar": "TestEnvironment"}
)

# Initialize the database manager
db_manager = MSSQLDB(config)

## 3. Start the Database

We'll now create and start the database. This process may take a bit longer for SQL Server as it's a more resource-intensive database engine:

In [ ]:
# Create and start the database
db_manager.create_db()
print(f"Database started successfully in container '{container_name}'")
print(f"Connection details:")
print(f"  Host: {config.host}")
print(f"  Port: {config.port}")
print(f"  User: {config.user}")
print(f"  Database: {config.database}")

## 4. Connect and Run SQL Queries

Now that our database is running, let's connect to it using SQL cell magic. For SQL Server, we'll use the ODBC connection string:

In [ ]:
%sqlcmd $config.database --password $config.password --username $config.user --driver "ODBC Driver 17 for SQL Server"  --server $config.host

This step executes code block `14` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
%%sqlcmd
SELECT * FROM test_table;

### Creating a Table

This subsection details `creating a table` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sqlcmd
CREATE TABLE demo_users (
    id INT IDENTITY(1,1) PRIMARY KEY,
    username VARCHAR(50) UNIQUE NOT NULL,
    email VARCHAR(100) UNIQUE,
    created_at DATETIME DEFAULT GETDATE()
);

### Insert Data

This subsection details `insert data` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sqlcmd
INSERT INTO demo_users (username, email) VALUES
    ('alice', 'alice@example.com'),
    ('bob', 'bob@example.com'),
    ('charlie', 'charlie@example.com');

### Query Data

This subsection details `query data` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sqlcmd
SELECT * FROM demo_users;

### Run More Complex Queries

This subsection details `run more complex queries` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sqlcmd
-- Create another table for demonstration
CREATE TABLE demo_posts (
    id INT IDENTITY(1,1) PRIMARY KEY,
    user_id INT FOREIGN KEY REFERENCES demo_users(id),
    title VARCHAR(100) NOT NULL,
    content NVARCHAR(MAX),
    created_at DATETIME DEFAULT GETDATE()
);

-- Insert some posts
INSERT INTO demo_posts (user_id, title, content) VALUES
    (1, 'Alice First Post', 'Hello world from Alice!'),
    (1, 'Alice Second Post', 'Another post from Alice'),
    (2, 'Bob Introduction', 'Hi, this is Bob!'),
    (3, 'Charlie Notes', 'Some notes from Charlie');
    
-- Query with JOIN
SELECT u.username, p.title, p.content
FROM demo_users u
JOIN demo_posts p ON u.id = p.user_id
ORDER BY u.username, p.created_at;

## 5. Using Regular Python to Access the Database

You can also interact with the database using Python code and pyodbc:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Connect directly using our db_manager's connection property
conn = db_manager.connection

# Create a cursor
cursor = conn.cursor()

# Execute a query
cursor.execute("SELECT COUNT(*) as post_count FROM demo_posts")
result = cursor.fetchone()
print(f"Total number of posts: {result[0]}")

# Group by query
cursor.execute("""
    SELECT u.username, COUNT(p.id) as post_count 
    FROM demo_users u
    LEFT JOIN demo_posts p ON u.id = p.user_id
    GROUP BY u.username
    ORDER BY post_count DESC
""")

print("\nPost count by user:")
for row in cursor.fetchall():
    print(f"  {row[0]}: {row[1]} posts")

# Close the connection
cursor.close()
conn.close()

## 6. Using SQL Server-Specific Features

Let's try some SQL Server-specific features like T-SQL stored procedures:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
%%sqlcmd
-- Create a stored procedure to get post count for a specific user
CREATE PROCEDURE GetUserPostCount
    @username VARCHAR(50)
AS
BEGIN
    SELECT u.username, COUNT(p.id) as post_count
    FROM demo_users u
    LEFT JOIN demo_posts p ON u.id = p.user_id
    WHERE u.username = @username
    GROUP BY u.username
END;

-- Execute the stored procedure
EXEC GetUserPostCount @username = 'alice';

## 7. Clean Up

When you're done with the database, you can delete it:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Delete the database container
db_manager.delete_db(running_ok=True)
print(f"Database container '{container_name}' deleted")

# Clean up the temporary directory
import shutil
shutil.rmtree(temp_dir)
print(f"Temporary directory '{temp_dir}' removed")

## 8. Conclusion

This notebook demonstrated how to:

1. Configure and create a Microsoft SQL Server database with `MSSQLDB`
2. Connect to the database using SQL cell magic with pyodbc
3. Execute SQL queries and create T-SQL stored procedures
4. Use pyodbc with Python to interact with the database
5. Clean up the database when finished

The `MSSQLDB` manager provides a convenient way to spin up SQL Server instances in Docker containers for development, testing, or demonstration purposes.

Note that SQL Server requires more resources than some other database engines, so ensure your Docker environment has sufficient memory and CPU allocated.

For production use, align this tutorial flow with your team standards: credential handling, migration strategy, backup policy, and monitoring. When needed, start from the provider setup docs (https://learn.microsoft.com/sql/linux/quickstart-install-connect-docker) and validate against real datasets before benchmarking query performance.